# STEP 3. 데이터 불러오기와 품질 점검

- 데이터: Google Trends "해외여행" 검색 관심도 (대한민국, 주간, 2021-11-07 ~ 2026-11-08)
- 값: 0~100 상대 지수 (실제 검색 횟수가 아님)
- 출처 표기: Data source: Google Trends (https://www.google.com/trends)

## 처리 기준 (STEP 3에서 확정)

| 구분 | 기준 | 이유 |
|---|---|---|
| 결측치 | 있으면 채우지 않고 개수와 위치를 기록한다 | 상대 지수라 보간값은 근거가 약하다 |
| 마지막 주 | 통계에서 제외하고 원본 파일에는 유지한다 | 수집일(2026-11-09)에 끝나지 않은 주일 수 있다 |
| 이상치 | 13주 이동 중앙값에서 크게 벗어난 주를 표시만 하고 삭제하지 않는다 | 급증이 실제 관심 변화일 수 있고, 추세가 있어 전체 분포 기준은 부적합하다 |

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

print("Python :", sys.version.split()[0])
print("numpy  :", np.__version__)
print("pandas :", pd.__version__)

Python : 3.13.17
numpy  : 2.4.6
pandas : 2.2.3


## 1. 불러오기

원본 CSV는 1행이 카테고리, 2행이 빈 줄, 3행이 열 이름이다. 앞 2줄을 건너뛰고 읽는다.

In [2]:
# 로컬에서는 data 폴더의 파일을, 파일이 없으면(Colab 등) GitHub의 원본 주소를 사용한다
CSV_PATH = Path("data/google_trends_overseas_travel_raw.csv")
RAW_URL = (
    "https://raw.githubusercontent.com/geunwa/"
    "overseas-travel-trend-analysis/main/data/google_trends_overseas_travel_raw.csv"
)
source = CSV_PATH if CSV_PATH.exists() else RAW_URL

raw = pd.read_csv(source, skiprows=2, encoding="utf-8-sig")
raw.columns = ["date", "value"]
raw["date"] = pd.to_datetime(raw["date"])
raw = raw.sort_values("date").reset_index(drop=True)

print("읽은 곳:", source)
print("행, 열 :", raw.shape)
print(raw.head(3))
print(raw.tail(3))

읽은 곳: data\google_trends_overseas_travel_raw.csv
행, 열 : (262, 2)
        date  value
0 2021-11-07     25
1 2021-11-14     22
2 2021-11-21     20
          date  value
259 2026-10-25     82
260 2026-11-01     77
261 2026-11-08     74


## 2. 품질 점검

결측치, 중복 날짜, 주 간격, 요일, 0값, 값의 범위를 확인한다.

In [3]:
gaps = raw["date"].diff().dt.days.dropna()

checks = pd.DataFrame(
    {
        "점검 항목": [
            "행 수",
            "결측치 수",
            "중복 날짜 수",
            "주 간격이 7일이 아닌 곳",
            "일요일이 아닌 날짜",
            "값이 0인 주",
            "값 최솟값",
            "값 최댓값",
        ],
        "결과": [
            len(raw),
            int(raw["value"].isna().sum()),
            int(raw["date"].duplicated().sum()),
            int((gaps != 7).sum()),
            int((raw["date"].dt.dayofweek != 6).sum()),
            int((raw["value"] == 0).sum()),
            int(raw["value"].min()),
            int(raw["value"].max()),
        ],
    }
)
print(checks.to_string(index=False))

         점검 항목  결과
           행 수 262
         결측치 수   0
       중복 날짜 수   0
주 간격이 7일이 아닌 곳   0
    일요일이 아닌 날짜   0
       값이 0인 주   0
         값 최솟값  11
         값 최댓값 100


## 3. 마지막 주 처리

수집일(2026-11-09)에 끝나지 않았을 수 있는 마지막 주(2026-11-08)를 분석용 데이터에서 제외한다. 원본 파일은 바꾸지 않는다.

In [4]:
raw["is_partial"] = raw["date"] == raw["date"].max()
data = raw[~raw["is_partial"]].drop(columns="is_partial").reset_index(drop=True)

print("전체 :", len(raw), "주")
print("분석 :", len(data), "주")
print("제외한 주")
print(raw.loc[raw["is_partial"], ["date", "value"]])

전체 : 262 주
분석 : 261 주
제외한 주
          date  value
261 2026-11-08     74


## 4. 이상치 표시

- 각 주의 값에서 앞뒤 13주(약 한 분기)의 중앙값을 뺀 편차를 구한다.
- 편차가 중앙값 절대 편차(MAD) 기준으로 크게 벗어나면(로버스트 z 점수 절댓값 3.5 초과) 이상치로 표시한다.
- 표시만 하고 삭제하지 않는다.

In [5]:
WINDOW = 13        # 13주 = 약 한 분기
THRESHOLD = 3.5    # 로버스트 z 점수 기준

data["median_13w"] = data["value"].rolling(WINDOW, center=True, min_periods=7).median()
data["deviation"] = data["value"] - data["median_13w"]

center = data["deviation"].median()
mad = (data["deviation"] - center).abs().median()
assert mad > 0, "MAD가 0이면 이상치 기준을 다시 정해야 한다"

data["robust_z"] = 0.6745 * (data["deviation"] - center) / mad
data["is_outlier"] = data["robust_z"].abs() > THRESHOLD

print("MAD :", mad)
print("이상치로 표시된 주:", int(data["is_outlier"].sum()), "개")
print(data.loc[data["is_outlier"], ["date", "value", "median_13w", "robust_z"]].round(2).to_string(index=False))

MAD : 2.0
이상치로 표시된 주: 4 개
      date  value  median_13w  robust_z
2024-12-01     60        49.0      3.71
2026-10-11     73        40.5     10.96
2026-10-18    100        41.0     19.90
2026-10-25     82        57.0      8.43


## 5. 분석용 데이터 저장

원본은 `data/google_trends_overseas_travel_raw.csv`에 그대로 두고, 처리 결과는 별도 파일로 저장한다.

In [6]:
Path("data").mkdir(exist_ok=True)
OUT_PATH = "data/weekly_clean.csv"
data.to_csv(OUT_PATH, index=False, encoding="utf-8-sig")
print("저장:", OUT_PATH, data.shape)

저장: data/weekly_clean.csv (261, 6)


In [7]:
# 이동평균(Moving Average): 최근 N주의 평균을 한 주씩 밀면서 구한다
# 4주 = 약 한 달의 짧은 흐름, 52주 = 1년의 긴 흐름(계절 변동이 평균에 섞여 사라진다)
series = data.set_index("date")["value"]

ma = pd.DataFrame(
    {
        "value": series,
        "ma_4w": series.rolling(4).mean(),
        "ma_52w": series.rolling(52).mean(),
    }
)

print("52주 평균이 처음 계산되는 날:", ma["ma_52w"].first_valid_index().date())
print(ma.tail(8).round(2))

52주 평균이 처음 계산되는 날: 2022-10-30
            value  ma_4w  ma_52w
date                            
2026-09-13     33  34.50   43.92
2026-09-20     40  35.75   43.85
2026-09-27     41  37.50   43.58
2026-10-04     34  37.00   43.31
2026-10-11     73  47.00   43.77
2026-10-18    100  62.00   44.88
2026-10-25     82  72.25   45.62
2026-11-01     77  83.00   46.15


In [8]:
# 달력 연도는 2021년(8주)과 2026년(45주)이 불완전하다
# 그래서 첫 주를 빼고 52주씩 5묶음으로 나눠 비교한다 (1묶음 = 약 1년)
blocks = data.iloc[1:].reset_index(drop=True).copy()
blocks["block"] = blocks.index // 52 + 1

summary = blocks.groupby("block").agg(
    start=("date", "min"),
    end=("date", "max"),
    weeks=("value", "size"),
    mean=("value", "mean"),
    median=("value", "median"),
    max=("value", "max"),
    outliers=("is_outlier", "sum"),
)
summary["growth_pct"] = summary["mean"].pct_change() * 100

# 이상치로 표시된 주를 뺀 평균 (급증이 결과를 얼마나 바꾸는지 확인용)
summary["mean_no_outlier"] = blocks[~blocks["is_outlier"]].groupby("block")["value"].mean()
summary["growth_no_outlier_pct"] = summary["mean_no_outlier"].pct_change() * 100

num_cols = ["mean", "median", "growth_pct", "mean_no_outlier", "growth_no_outlier_pct"]
summary[num_cols] = summary[num_cols].round(2)
print(summary.to_string())

total_change = (summary["mean"].iloc[-1] / summary["mean"].iloc[0] - 1) * 100
print("\n첫 묶음 대비 마지막 묶음 변화율:", round(total_change, 1), "%")

           start        end  weeks   mean  median  max  outliers  growth_pct  mean_no_outlier  growth_no_outlier_pct
block                                                                                                               
1     2021-11-14 2022-11-06     52  27.62    32.0   39         0         NaN            27.62                    NaN
2     2022-11-13 2023-11-05     52  39.83    39.0   55         0       44.22            39.83                  44.22
3     2023-11-12 2024-11-03     52  38.81    39.0   49         0       -2.56            38.81                  -2.56
4     2024-11-10 2025-11-02     52  46.23    46.5   60         1       19.13            45.96                  18.43
5     2025-11-09 2026-11-01     52  46.15    41.0  100         3       -0.17            43.78                  -4.75

첫 묶음 대비 마지막 묶음 변화율: 67.1 %


In [9]:
ma52 = ma["ma_52w"].dropna()
peak_date = ma52.idxmax()
before_spike = ma52.loc["2026-09-27"]

print("52주 평균 처음  :", round(ma52.iloc[0], 2), ma52.index[0].date())
print("52주 평균 최고  :", round(ma52.max(), 2), peak_date.date())
print("52주 평균 9/27  :", round(before_spike, 2))
print("52주 평균 마지막:", round(ma52.iloc[-1], 2), ma52.index[-1].date())
print("최고점 대비 9/27 변화율:", round((before_spike / ma52.max() - 1) * 100, 1), "%")

52주 평균 처음  : 27.35 2022-10-30
52주 평균 최고  : 47.35 2026-04-05
52주 평균 9/27  : 43.58
52주 평균 마지막: 46.15 2026-11-01
최고점 대비 9/27 변화율: -8.0 %


In [10]:
# 전주 대비 변화: 이번 주 값에서 지난주 값을 뺀 값(diff)과 변화율(pct_change)
change = pd.DataFrame(
    {
        "value": series,
        "diff": series.diff(),
        "pct": series.pct_change() * 100,
    }
)
print(change.tail(8).round(2))

# 평소 변동의 크기: 주간 변화율의 평균과 표준편차 (260주)
pct = change["pct"].dropna()
print("\n주간 변화율 평균:", round(pct.mean(), 2), "%  표준편차:", round(pct.std(), 2), "%")

# 변화폭(절대값)이 큰 주 5개
top = change["diff"].abs().sort_values(ascending=False).head(5).index
print("\n변화폭 상위 5주")
print(change.loc[top, ["value", "diff", "pct"]].round(2))

            value  diff     pct
date                           
2026-09-13     33  -3.0   -8.33
2026-09-20     40   7.0   21.21
2026-09-27     41   1.0    2.50
2026-10-04     34  -7.0  -17.07
2026-10-11     73  39.0  114.71
2026-10-18    100  27.0   36.99
2026-10-25     82 -18.0  -18.00
2026-11-01     77  -5.0   -6.10

주간 변화율 평균: 1.2 %  표준편차: 13.32 %

변화폭 상위 5주
            value  diff     pct
date                           
2026-10-11     73  39.0  114.71
2026-10-18    100  27.0   36.99
2026-10-25     82 -18.0  -18.00
2026-02-08     44 -13.0  -22.81
2024-12-01     60  12.0   25.00


In [11]:
# 급증 전 13주(기준선)와 급증 후 4주를 비교한다
baseline = series["2026-07-12":"2026-10-04"]
after = series["2026-10-11":]
base_mean = baseline.mean()

print("기준선 13주 평균:", round(base_mean, 2), " 중앙값:", baseline.median())
print("급증 후 4주 평균:", round(after.mean(), 2))
print("기준선 대비 배수:", round(after.mean() / base_mean, 2), "배")

# 정점(10-18) 이후 얼마나 남아 있는가: 잔존율 = (값 - 기준선) / (정점 - 기준선)
peak = series.loc["2026-10-18"]
post = series["2026-10-18":]
last_week = raw[raw["is_partial"]].set_index("date")["value"]  # 통계에서 뺀 마지막 주
post = pd.concat([post, last_week])

result = pd.DataFrame(
    {
        "value": post,
        "excess": (post - base_mean).round(2),
        "retention": ((post - base_mean) / (peak - base_mean)).round(3),
    }
)
result["note"] = ["정점", "", "", "미완성 가능, 통계 제외"]
print()
print(result)

기준선 13주 평균: 36.15  중앙값: 35.0
급증 후 4주 평균: 83.0
기준선 대비 배수: 2.3 배

            value  excess  retention           note
date                                               
2026-10-18    100   63.85      1.000             정점
2026-10-25     82   45.85      0.718               
2026-11-01     77   40.85      0.640               
2026-11-08     74   37.85      0.593  미완성 가능, 통계 제외


In [12]:
# 비교 사례: 2024-12-01 이상치(60) 전후
spike = pd.Timestamp("2024-12-01")
before4 = series[spike - pd.Timedelta(weeks=4) : spike - pd.Timedelta(weeks=1)]
after8 = series[spike + pd.Timedelta(weeks=1) : spike + pd.Timedelta(weeks=8)]

print("급증 전 4주 평균:", round(before4.mean(), 2))
print("급증 값        :", series.loc[spike])
print("급증 후 8주 평균:", round(after8.mean(), 2))
retention_2024 = (after8.mean() - before4.mean()) / (series.loc[spike] - before4.mean())
print("후 8주 평균의 잔존율:", round(retention_2024, 3))

급증 전 4주 평균: 42.25
급증 값        : 60
급증 후 8주 평균: 53.25
후 8주 평균의 잔존율: 0.62
